In [16]:
import matplotlib.pyplot as plt
from tqdm.auto import trange
from scipy.stats import qmc
import numpy as np
import os

In [17]:
"""
Fractional release profile from a sphere (Crank's diffusion equation).

    M_t / M_inf = 1 - (6/pi^2) * sum_{n=1}^{inf} (1/n^2) * exp(-D * n^2 * pi^2 * t / R^2)
"""

def fractional_release(t, D, R, n_terms=50):
    """
    Compute M_t / M_inf for diffusion out of a sphere.

    Parameters
    ----------
    t : float or array_like
        Time(s) at which to evaluate the release. Same time units as implied by D.
    D : float
        Diffusion coefficient (e.g. m^2/s). Must be > 0.
    R : float
        Sphere radius (same length units as D, e.g. m). Must be > 0.
    n_terms : int, optional

    Returns
    -------
    Mt_over_Minf : ndarray
        Fractional mass released, in [0, 1]. Same shape as `t`.
    """
    if D <= 0 or R <= 0:
        raise ValueError("D and R must be positive.")
    if n_terms < 1:
        raise ValueError("n_terms must be >= 1.")

    t = np.asarray(t, dtype=float)

    # n = 1, 2, ..., n_terms  -> shape (n_terms,)
    n = np.arange(1, n_terms + 1)

    # Broadcast: t has shape (...,), n has shape (n_terms,)
    # exponent has shape (..., n_terms)
    exponent = -D * (n**2) * (np.pi**2) * t[..., np.newaxis] / (R**2)
    series = np.sum(np.exp(exponent) / n**2, axis=-1)

    result = 1.0 - (6.0 / np.pi**2) * series

    # At t = 0 the series sums to pi^2/6 exactly, giving 0.0 — but clip tiny
    # negative values from floating-point error.
    return np.clip(result, 0.0, 1.0)

# Generate data

## TRAINING DATA

### Generate Training Data

In [18]:
# ==========================================
# 1. THE SOBOL ENGINE
# ==========================================
N_cases = 8192 # Must be a power of 2 for Sobol
# N_cases = 1024 # Must be a power of 2 for Sobol
d_dimensions = 5 # R, epsilon, tau_ort, D0, Fo_final

sampler = qmc.Sobol(d=d_dimensions, scramble=True)
u = sampler.random(n=N_cases)

# ==========================================
# 2. PARAMETER MAPPING
# ==========================================

# A. Radius (Logarithmic: 0.5 to 50 microns -> Convert to cm for consistency)
# 0.5 um = 5e-5 cm | 50 um = 5e-3 cm
log_R_min, log_R_max = np.log10(5e-5), np.log10(5e-3)
R_cm = 10 ** (log_R_min + u[:, 0] * (log_R_max - log_R_min))

# # B. Porosity (Linear: 0.05 to 0.50)
epsilon = 0.05 + u[:, 1] * (0.50 - 0.05)

# # C. Tortuosity (Linear: 1.5 to 5.0)
tau_ort = 1.5 + u[:, 2] * (5.0 - 1.5)

# # D. Base Diffusion D0 (Logarithmic: min to max cm^2/s)
log_D0_min, log_D0_max = np.log10(1e-12), np.log10(1.5e-11)
D0 = 10 ** (log_D0_min + u[:, 3] * (log_D0_max - log_D0_min))

# # Calculate the Effective Diffusion for the physical simulation
D_eff = D0 * (epsilon / tau_ort) 

# ==========================================
# 3. THE TIME DOMAIN
# ==========================================

# # Target Fourier Number (Linear: 0.05 to 0.65)
Fo_final = 0.1 + u[:, 4] * (0.65 - 0.05)

# # Calculate the exact Absolute Time (t_end in seconds) for every single case
t_end_seconds = (Fo_final * (R_cm ** 2)) / D_eff

# # The Dimensionless Time Vector (This is what you feed the PINN Trunk)
tau_vector = np.linspace(0, 1, 200) ** 2

In [ ]:
c_release = np.empty((N_cases, 200))
t_release = np.empty((N_cases, 200))


max_release_distribution = []


for i in trange(N_cases):
    R = R_cm[i]
    D = D_eff[i]
    
    # # Characteristic diffusion time tau = R^2 / D
    time_final = R**2 / D * Fo_final[i]
    
    times = tau_vector * time_final
    
    fr = fractional_release(times, D, R, n_terms=170)

    c_release[i, :] = fr
    t_release[i, :] = times

    plt.plot(times, fr)

    max_release_distribution.append(fr[-1])
    # break

### Save Training Data

In [20]:
master_directory = "datasetA_training_data"
os.makedirs(master_directory, exist_ok=True)


np.save(f"{master_directory}/R_array.npy", R_cm) # cm
np.save(f"{master_directory}/epsilon_array.npy", epsilon) # cm
np.save(f"{master_directory}/tau_ort_array.npy", tau_ort) # cm
np.save(f"{master_directory}/D0_array.npy", D0) # cm
np.save(f"{master_directory}/Deff_array.npy", D_eff) # cm

np.save(f"{master_directory}/Fo_final_array.npy", Fo_final) # cm
np.save(f"{master_directory}/t_end_seconds_array.npy", t_end_seconds) # cm


np.save(f"{master_directory}/c_release_matrix.npy", c_release) # cm
np.save(f"{master_directory}/t_release_matrix.npy", t_release) # cm

## TESTING DATA

### Generate Testing Data

In [22]:
# ==========================================
# 1. THE SOBOL ENGINE
# ==========================================
N_cases = 1024 # Must be a power of 2 for Sobol
d_dimensions = 5 # R, epsilon, tau_ort, D0, Fo_final

sampler = qmc.Sobol(d=d_dimensions, scramble=True)
u = sampler.random(n=N_cases)

# ==========================================
# 2. PARAMETER MAPPING
# ==========================================

# A. Radius (Logarithmic: 0.5 to 50 microns -> Convert to cm for consistency)
# 0.5 um = 5e-5 cm | 50 um = 5e-3 cm
log_R_min, log_R_max = np.log10(5e-5), np.log10(5e-3)
R_cm = 10 ** (log_R_min + u[:, 0] * (log_R_max - log_R_min))

# # B. Porosity (Linear: 0.05 to 0.50)
epsilon = 0.05 + u[:, 1] * (0.50 - 0.05)

# # C. Tortuosity (Linear: 1.5 to 5.0)
tau_ort = 1.5 + u[:, 2] * (5.0 - 1.5)

# # D. Base Diffusion D0 (Logarithmic: min to max cm^2/s)
log_D0_min, log_D0_max = np.log10(1e-12), np.log10(1.5e-11)
D0 = 10 ** (log_D0_min + u[:, 3] * (log_D0_max - log_D0_min))

# # Calculate the Effective Diffusion for the physical simulation
D_eff = D0 * (epsilon / tau_ort) 

# ==========================================
# 3. THE TIME DOMAIN
# ==========================================

# # Target Fourier Number (Linear: 0.05 to 0.65)
Fo_final = 0.1 + u[:, 4] * (0.65 - 0.05)

# # Calculate the exact Absolute Time (t_end in seconds) for every single case
t_end_seconds = (Fo_final * (R_cm ** 2)) / D_eff

# # The Dimensionless Time Vector (This is what you feed the PINN Trunk)
tau_vector = np.linspace(0, 1, 200) ** 2

In [ ]:
c_release = np.empty((N_cases, 200))
t_release = np.empty((N_cases, 200))


max_release_distribution = []


for i in trange(N_cases):
    R = R_cm[i]
    D = D_eff[i]
    
    # # Characteristic diffusion time tau = R^2 / D
    time_final = R**2 / D * Fo_final[i]
    
    times = tau_vector * time_final
    
    fr = fractional_release(times, D, R, n_terms=170)

    c_release[i, :] = fr
    t_release[i, :] = times

    plt.plot(times, fr)

    max_release_distribution.append(fr[-1])
    # break

### Save Testing Data

In [24]:
master_directory = "datasetA_testing_data"
os.makedirs(master_directory, exist_ok=True)


np.save(f"{master_directory}/R_array.npy", R_cm) # cm
np.save(f"{master_directory}/epsilon_array.npy", epsilon) # cm
np.save(f"{master_directory}/tau_ort_array.npy", tau_ort) # cm
np.save(f"{master_directory}/D0_array.npy", D0) # cm
np.save(f"{master_directory}/Deff_array.npy", D_eff) # cm

np.save(f"{master_directory}/Fo_final_array.npy", Fo_final) # cm
np.save(f"{master_directory}/t_end_seconds_array.npy", t_end_seconds) # cm


np.save(f"{master_directory}/c_release_matrix.npy", c_release) # cm
np.save(f"{master_directory}/t_release_matrix.npy", t_release) # cm